![MSE Logo](https://moodle.msengineering.ch/pluginfile.php/1/core_admin/logo/0x150/1643104191/logo-mse.png)

# AdvNLP Lab (Graded Lab): Experimenting with Retrieval as Part of a RAG System

Total: 44 points

**Objectives:** We build the retrieval part of a RAG system and compare performance of classic KNN retrieval with additional cross encoder reranking. Eventually, we write two prompts for generation and test it on a LLM.

**Useful documentation:** Since you'll use LangChain for this assignment, [their documentation](https://python.langchain.com/docs/introduction/) might be helpful.

## Students

Gil Cordeiro Diego Manuel, Rajilatha Kandiah

## Setup

First, we need to install the required packages for this assignment.

In [ ]:
# %pip install pandas langchain langchain-classic langchain-core langchain-community langchain-classic langchain-huggingface faiss-cpu --quiet

In [1]:
import pandas as pd
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker

c:\Users\Rajilatha.Kandiah\AppData\Local\miniconda3\envs\AdvNLP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


We will use the [DRAGONBall Dataset](https://github.com/OpenBMB/RAGEval) as a basis for this assignment and load a subset of their documents. These will be the stored knowledge of the RAG system. To store them into the vector store, we will later directly create embeddings out of them, since they have alredy the size of suitable chunks. Each document consists of a unique ID and the actual content.

In [2]:
documents = pd.read_csv('docs.csv', index_col=0)
documents

,content
id,
40,Acme Government Solutions is a government indu...
41,Entertainment Enterprises Inc. is an entertain...
42,"Advanced Manufacturing Solutions Inc., establi..."
43,"EcoGuard Solutions, established on April 15, 2..."
44,"Green Fields Agriculture Ltd., established on ..."
...,...
211,Hospitalization Record:\n\nBasic Information:\...
212,**Hospitalization Record**\n\n**Basic Informat...
213,Hospitalization Record\n\nBasic Information:\n...


The main goal of the assignment is to evaluate the retrieval component of the RAG system. For that, we also load a dataset of queries, which we can use to retrieve matching documents. Each query has also assigned an array of documents in the form of their IDs, which match with the documents loaded before. We can use these to evaluate whether the correct documents were found by the retrieval or not.

In [3]:
queries = pd.read_csv('queries.csv', index_col=0)
queries['ground_truth_doc_ids'] = queries['ground_truth_doc_ids'].apply(lambda x: x.split(';'))
queries

,query,ground_truth_doc_ids
query_id,,
2286,When was Sparkling Clean Housekeeping Services...,[64]
2433,How did HealthPro Innovations' strategic partn...,[54]
6266,According to the hospitalization records of Br...,[212]
4499,"According to the judgment of Norwood, Unionvil...",[124]
2448,Based on HealthLife Solutions' 2020 corporate ...,[73]
...,...,...
2186,How did the severe drought in August 2018 lead...,[65]
3251,Compare the large-scale financing activities o...,"[58, 55]"
2268,How did CleanCo Housekeeping Services' investm...,[47]


## 1. Recall@N

**1a) [2 points]** We will evaluate the retrieval by comparing the retrieved documents with the ground truth documents assigned to the query. For that, we will use the Recall@N metric. Please describe in 1-2 sentences how we can interpret this metric in our case.

**Your Answer:**

- Recall@N is an evaluation metric which measures how many of the relevant documents were found by the retriever in the top N results. It is calculated by dividing the count of relevant documents in top N by the total relevant document count. 

**1b) [4 points]** Implement the Recall@N metric and test it with the following code.

In [4]:
def recall_at_n(retrieved_docs, relevant_doc_ids, n):
    """
    Calculate Recall@N.

    Parameters:
    - retrieved_docs: Sorted list of retrieved documents as LangChain Document objects
    - relevant_doc_ids: List of relevant document IDs
    - n: Number of top documents to consider

    Returns:
    - Recall@N
    """
    top_n = retrieved_docs[:n]
    top_n_ids = [str(d.metadata.get('id')) for d in top_n]
    hits = sum(1 for rid in relevant_doc_ids if str(rid) in top_n_ids)
    return hits / len(relevant_doc_ids)

In [5]:
### Test

recall_at_n(
    [Document(page_content='', metadata={'id': str(id)}) for id in range(10)],
    ['0', '1', '20'],
    3
)

0.6666666666666666

## 2. Embedding Model

**2a) [3 points]** Each document will be converted to an embedding representing the semantic meaning of the document. In this assignment, we will use model `sentence-transformers/all-MiniLM-L6-v2` from HuggingFace. Please answer the following questions about this model:

**Your Answers:**

Embedding Length: 384

Number of Parameters: 22'713'216

Maximum Sequence Length: 256

In [6]:
# %pip install sentence-transformers
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = model.encode(["This is an example sentence"])

print(f"embedding Length:        {embeddings.shape[1]}")        
print(f"maximum sequence length: {model.get_max_seq_length()}")
print(f"num of parameters:    {sum(p.numel() for p in model.parameters()):,}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6658.34it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding Length:        384
maximum sequence length: 256
num of parameters:    22,713,216


## 3. Vector Store

**3a) [4 points]** Use LangChain to create a FAISS vector store and embed the documents with the above-mentioned embedding model. Load the documents again but this time with a Loader object from LangChain. Eventually, print the number of documents in the vector store.

In [7]:
from langchain_community.document_loaders import CSVLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

#load documents from CSV with langchain loader
loader = CSVLoader('docs.csv', source_column='id', metadata_columns=['id'])
langchain_docs = loader.load()
print(f"Loaded documents: {len(langchain_docs)}")

#initialize HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

# create FAISS vector store
vector_store = FAISS.from_documents(langchain_docs, embedding_model)
print(f"Number of documents in vector store: {vector_store.index.ntotal}") 


Loaded documents: 108


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4129.55it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Number of documents in vector store: 108


**3b) [3 points]** Retrieve the Top-3 documents for this query: "According to the hospitalization records of Bridgewater General Hospital, summarize the present illness of J. Reyes." and print the documents' ID and L2 distance.

In [8]:
query_3b = "According to the hospitalization records of Bridgewater General Hospital, summarize the present illness of J. Reyes."

retrieved_docs = vector_store.similarity_search_with_score(query_3b, k=3)
retrieved_docs_sorted = sorted(retrieved_docs, key=lambda x: x[1])

print("Top 3 retrieved Documents:")
for doc, score in retrieved_docs_sorted:
    print(f"ID: {doc.metadata['id']},  L2-Distanz: {score:.4f}")


Top 3 retrieved Documents:
ID: 212,  L2-Distanz: 0.7302
ID: 213,  L2-Distanz: 0.9898
ID: 210,  L2-Distanz: 1.0050


**3c) [2 points]** Check and show if a suitable document is found for the query in the Top-3 retrieved documents and show the relevant ones.

In [9]:
query_3b_id = 6266
gt_doc_ids = queries.loc[query_3b_id, 'ground_truth_doc_ids']

print("Top-3 retrieved documents:")
for doc, score in retrieved_docs_sorted:
    doc_id = doc.metadata['id']
    is_relevant = str(doc_id) in [str(g) for g in gt_doc_ids]
    print(f"  ID: {doc_id}, L2 distance: {score:.4f}, relevant: {is_relevant}")

relevant_in_top3 = [
    doc for doc, _ in retrieved_docs_sorted
    if str(doc.metadata['id']) in [str(g) for g in gt_doc_ids]
]

print(f"\nRelevant documents found in Top-3: {len(relevant_in_top3)} / {len(gt_doc_ids)}")
for doc in relevant_in_top3:
    print(f"\nRelevant document ID: {doc.metadata['id']}")
    print(doc.page_content[:500])


Most relevant document (Rank 1):
ID: 212
content: **Hospitalization Record**

**Basic Information:**
Name: J. Reyes
Gender: Male
Age: 52
Ethnicity: Hispanic
Marital Status: Married
Occupation: Construction Worker
Address: 22, Sunnyvale street, Bridgewater
Admission Time: 7th, September
Record Time: 8th, September
Historian: Self
Hospital Name: Bridgewater General Hospital

**Chief Complaint:**
Persistent joint pain and morning stiffness for 6 months

**Present Illness:**
Onset: The symptoms began insidiously 6 months ago, initially not


**Your Answer:**

The top-1 retrieved document (ID: 212) with the lowest L2 distance contains the hospitalization records of J. Reyes at Bridgewater General Hospital and seems therefore a suitable document for this query. This shows that the embedding model successfully identified the correct document for the query.


## 4. Vector Store Evaluation

**4a) [4 points]** Now, we will search with each of the queries for the most relevant documents in the vector store, and calculate Recall@N with them and the assigned ground truth document IDs. To aggregate the results over all queries, we will calculate the mean. We will do this 3 times to and use a different value for $N$ each time: $N \in \{ 1, 3, 5, 25\}$.

In [10]:
n_values = [1,3,5,25]
n_max = max(n_values)

all_retrieved = []
for _, row in queries.iterrows():
    retrieved = vector_store.similarity_search(row['query'], k=n_max)
    all_retrieved.append((retrieved, row['ground_truth_doc_ids']))

# Mean recall for all queries
for n in n_values:
    recalls = [
        recall_at_n(retrieved, gt_ids, n)
        for retrieved, gt_ids in all_retrieved
    ]
    print(f"  Recall for n {n:2d} = {sum(recalls) / len(recalls):.4f}")

  Recall for n  1 = 0.6650
  Recall for n  3 = 0.8150
  Recall for n  5 = 0.8600
  Recall for n 25 = 1.0000


**4b) [2 points]** When looking at the four calculated Recall@N scores, what do you observe and how can you explain this?

**Your Answer:**

We can observe that the higher the N, the higher the recall is, which makes sense since considering more documents can only include more relevant ones. At N
n=1 only 66% of the queries have the correct document at rank 1, meaning the embedding model does not always place the correct document to the top. 
However at n=25 the recall reaches 1.0, meaning the correct document is always somewhere in the top 25. 


## 5. Cross Encoder

**5a) [3 points]** We want to use a cross encoder model to rerank the retrieved documents. Describe in 1-2 sentences how a new document order can be determined using a cross encoder.

**Your Answer:**

A cross encoder concatenates the query and the candidate document into a single input sequence and processes it with a pre-trained BERT model with an additional classification head, applying full self-attention across all tokens. This gives a relevance score for each query-document pair which is then used to rerank the top N candidates.

**5b) [4 points]** Now again, we want to calculate Recall@N for all queries and the same $N$ as before. This time, we want to rerank the Top-25 retrieved documents using the cross encoder model `BAAI/bge-reranker-base`. Implement this using LangChain components and report the average Recall for $N \in \{ 1, 3, 5, 25\}$.

In [12]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_classic.retrievers import ContextualCompressionRetriever

cross_encoder = HuggingFaceCrossEncoder(model_name='BAAI/bge-reranker-base')
reranker = CrossEncoderReranker(model=cross_encoder, top_n=n_max)

# load top 25
base_retriever = vector_store.as_retriever(search_kwargs={'k': n_max})

# wrap with retreiver that reranks the top 25
compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=base_retriever
)

# rerank queries
all_reranked = []
for _, row in queries.iterrows():
    reranked_docs = compression_retriever.invoke(row['query'])
    all_reranked.append((reranked_docs, row['ground_truth_doc_ids']))

# calculate recall for reranked results
print("Mean Recall with Cross Encoder Reranking ")
for n in n_values:
    recalls = [
        recall_at_n(reranked, gt_ids, n)
        for reranked, gt_ids in all_reranked
    ]
    print(f"  Recall for n {n:2d} = {sum(recalls) / len(recalls):.4f}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6487.98it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Mean Recall with Cross Encoder Reranking 
  Recall for n  1 = 0.7400
  Recall for n  3 = 0.9600
  Recall for n  5 = 0.9800
  Recall for n 25 = 1.0000


**5c) [2 points]** What do you observe when you compare the Recall@N scores after reranking with the scores without reranking? Write 1-2 sentences about this and why this might happen.

**Your Answer:**

We can see that recall@1, @3 and @5 all improved significantly compared to the FAISS model.
Recall@25 stays the same since the same 25 candidates are just reordered and not filtered. This shows that the cross encoder is much better at ranking the correct document to the top compared to FAISS alone, since it evaluates the full query-document interaction instead of just comparing independent embeddings.

## 6. Generation

**6a) [6 points]** After improving the retrieval part of the RAG system, we want to finally generate an answer for our query. Retrieve the most relevant document for query "How much funding did HealthPro Innovations raise in February 2021?" and print its ID. Then write the instruction message of a prompt to answer this query including all necessary elements before running it using your favourite LLM (ChatGPT GPT-4o, etc.). Please paste the answer from the model and indicate which model you used.

In [11]:
query_6a = "How much funding did HealthPro Innovations raise in February 2021?"

# retrieve most relevant document
top_doc = vector_store.similarity_search(query_6a, k=1)[0]
print(f"Retrieved document ID: {top_doc.metadata['id']}")
print(f"\nDocument content:\n{top_doc.page_content}")


Retrieved document ID: 54

Document content:
content: HealthPro Innovations, established in September 2009, is a publicly traded healthcare company based in New York, specializing in the development and sale of innovative healthcare solutions.
In 2021, HealthPro Innovations experienced several significant events that had a profound impact on its financial performance and market position. Firstly, in February 2021, the company conducted a large-scale financing activity, raising $150 million in funds. This financing activity strengthened the company's financial strength and provided support for its expansion and development. One of the sub-events that contributed to this financing activity was the strategic partnership formed with a leading healthcare provider in January 2021. This partnership aimed to expand HealthPro Innovations' market reach and gain access to new customers, ultimately resulting in increased sales and revenue potential.
In March 2021, HealthPro Innovations successfull

**Your Prompt:**

You are a financial analyst assistant. Answer the question strictly based 
on the provided context. Do not use any external knowledge.

Context:
HealthPro Innovations, established in September 2009, is a publicly traded healthcare company based in New York, specializing in the development and sale of innovative healthcare solutions. In 2021, HealthPro Innovations experienced several significant events that had a profound impact on its financial performance and market position. Firstly, in February 2021, the company conducted a large-scale financing activity, raising $150 million in funds. This financing activity strengthened the company's financial strength and provided support for its expansion and development. One of the sub-events that contributed to this financing activity was the strategic partnership formed with a leading healthcare provider in January 2021. This partnership aimed to expand HealthPro Innovations' market reach and gain access to new customers, ultimately resulting in increased sales and revenue potential. In March 2021, HealthPro Innovations successfully launched a new line of innovative healthcare solutions, offering advanced features and improved patient outcomes. This product launch stimulated customer interest and generated additional revenue streams for the company. Additionally, in April 2021, HealthPro Innovations acquired a promising research and development startup specializing in cutting-edge medical technology. This acquisition enhanced the company's research capabilities, accelerating the development of new and improved healthcare solutions.

Question:
How much funding did HealthPro Innovations raise in February 2021?

**Generated Answer:** 

In February 2021, HealthPro Innovations raised $150 million in funds through a large-scale financing activity.


**Used Model:**

Chat GPT Modell: GPT-5.5 Instant


**6b) [3 points]** We want to use in-context learning and provide the LLM one example of a possible answer. Use the same prompt and extend it, that it should follow this example answer: "Yep, they sold a lot in that year. Over 50 million units as I can see — pretty big move, respect!". Use the same model, create a fresh chat and run this new prompt. Highlight the changes in the prompt using **bold style** or <span style="color:red;">color</span>.

**Your Prompt:**

You are a financial analyst assistant. Answer the question strictly based on the provided context. Do not use any external knowledge. **Here is an example of how you should answer: Question: How many units did TechCorp sell in 2020? Answer: "Yep, they sold a lot in that year. Over 50 million units as I can see — pretty big move, respect!" Now answer the following question in the same style.**

Context:
HealthPro Innovations, established in September 2009, is a publicly traded healthcare company based in New York, specializing in the development and sale of innovative healthcare solutions. In 2021, HealthPro Innovations experienced several significant events that had a profound impact on its financial performance and market position. Firstly, in February 2021, the company conducted a large-scale financing activity, raising $150 million in funds. This financing activity strengthened the company's financial strength and provided support for its expansion and development. One of the sub-events that contributed to this financing activity was the strategic partnership formed with a leading healthcare provider in January 2021. This partnership aimed to expand HealthPro Innovations' market reach and gain access to new customers, ultimately resulting in increased sales and revenue potential. In March 2021, HealthPro Innovations successfully launched a new line of innovative healthcare solutions, offering advanced features and improved patient outcomes. This product launch stimulated customer interest and generated additional revenue streams for the company. Additionally, in April 2021, HealthPro Innovations acquired a promising research and development startup specializing in cutting-edge medical technology. This acquisition enhanced the company's research capabilities, accelerating the development of new and improved healthcare solutions.

Question: How much funding did HealthPro Innovations raise in February 2021? 

**Generated Answer:**

Yep, they made a pretty serious move that month. In February 2021, HealthPro Innovations raised $150 million in funding as I can see — strong capital boost, respect!


**6c) [2 points]** Please check if the two answers are correct according to the document and how they differ. Does the model follow the example in the second prompt?

**Your Answer:**

Both answers are factually correct. HealthPro Innovations raised $150 million in February 2021 according to document ID 54. However they differ in style. The first answer is formal and precise while the second adopts the casual tone of the provided example. Yes, the model follows the example in the second prompt which shows that one-shot in-context learning effectively controls the output style without any fine-tuning.


## End of AdvNLP Lab

Please make sure all cells have been executed, save this completed notebook, and upload it to Moodle.